<a href="https://colab.research.google.com/github/harini200614/Data-Visualization-lab/blob/main/DVT_Exp4__231401033.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DVT EXPERIMENT 4 – Data Selection, Sorting, Statistics & Grouping



In [2]:


import os
import glob
import zipfile
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATASET_URL = "https://www.kaggle.com/api/v1/datasets/download/adrianjuliusaluoch/daily-google-search-trends-us"
ZIP_PATH = "/content/google_trends_kaggle.zip"
DATA_DIR = "/content/google_trends_kaggle"

# Download only if the dataset is not already present
if not os.path.exists(DATA_DIR):
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
    os.makedirs(DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_DIR)

csv_files = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)

if not csv_files:
    raise FileNotFoundError("Kaggle CSV was not found after download.")

CSV_PATH = csv_files[0]
df = pd.read_csv(CSV_PATH)

print("Kaggle Google Trends dataset loaded successfully.")
print("CSV:", CSV_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())


Kaggle Google Trends dataset loaded successfully.
CSV: /content/google_trends_kaggle/trending_searches_in_us.csv
Shape: (31804, 8)
Columns: ['query', 'start_date', 'end_date', 'active', 'search_volume', 'increase_percentage', 'categories', 'trend_breakdown']


,query,start_date,end_date,active,search_volume,increase_percentage,categories,trend_breakdown
0,shedeur sanders,2026-03-30 18:40:00+00:00,NaN,True,200,50,Sports,NaN
1,whoopi goldberg,2026-03-30 18:40:00+00:00,NaN,True,5000,50,Politics,"america first award, mike johnson"
2,mo williams,2026-03-30 18:00:00+00:00,NaN,True,5000,300,Sports,"mason williams basketball, mason williams"
3,sloane stephens,2026-03-30 17:50:00+00:00,NaN,True,1000,200,"Sports, Health",NaN
4,canceled tv shows 2026,2026-03-30 17:50:00+00:00,NaN,True,5000,200,Entertainment,renewed and cancelled tv shows 2026


In [3]:
# Prepare the real Kaggle Google Trends columns safely
df.columns = (
    df.columns.astype(str)
      .str.strip()
      .str.lower()
      .str.replace(" ", "_", regex=False)
)

def find_col(names):
    for name in names:
        if name in df.columns:
            return name
    return None

query_col = find_col(["query", "trends", "trend", "search_query", "keyword", "term", "search_term"])
date_col = find_col(["date", "collection_date", "start_time", "date_recorded", "datetime", "timestamp", "time"])
location_col = find_col(["location", "country", "region", "geo"])
volume_col = find_col(["search_volume_lower", "search_volume", "volume"])

if query_col is None:
    raise ValueError("Search-query column not found. Available columns: " + str(df.columns.tolist()))

if date_col is not None:
    df["date"] = pd.to_datetime(df[date_col], errors="coerce")
    df["year"] = df["date"].dt.year
else:
    df["date"] = pd.NaT
    df["year"] = np.nan

df["query"] = df[query_col].astype(str).str.strip()

def to_volume(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().upper().replace(",", "").replace("+", "")
    try:
        if s.endswith("B"): return float(s[:-1]) * 1_000_000_000
        if s.endswith("M"): return float(s[:-1]) * 1_000_000
        if s.endswith("K"): return float(s[:-1]) * 1_000
        return float(s)
    except ValueError:
        return np.nan

if volume_col is not None:
    df["search_volume"] = df[volume_col].apply(to_volume)
else:
    df["search_volume"] = df.groupby("query")["query"].transform("count").astype(float)

df["location"] = (
    df[location_col].astype(str).str.strip()
    if location_col is not None
    else "United States"
)

df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.reset_index(drop=True)

print("Prepared columns: query, date, year, location, search_volume")
display(df[["query", "date", "year", "location", "search_volume"]].head())


Prepared columns: query, date, year, location, search_volume


,query,date,year,location,search_volume
0,shedeur sanders,NaT,NaN,United States,200.0
1,whoopi goldberg,NaT,NaN,United States,5000.0
2,mo williams,NaT,NaN,United States,5000.0
3,sloane stephens,NaT,NaN,United States,1000.0
4,canceled tv shows 2026,NaT,NaN,United States,5000.0


## 1. Shape

In [4]:
print('Shape:',df.shape); print('Rows:',df.shape[0]); print('Columns:',df.shape[1])

Shape: (31804, 11)
Rows: 31804
Columns: 11


## 2. Data Types

In [5]:
print(df.dtypes)

query                          object
start_date                     object
end_date                       object
active                           bool
search_volume                 float64
increase_percentage             int64
categories                     object
trend_breakdown                object
date                   datetime64[ns]
year                          float64
location                       object
dtype: object


## 3. Filter Recent Trends

In [6]:
recent=df[df['year']>2020]; print('Records after 2020:',len(recent)); display(recent[['query','date','year','location','search_volume']].head(20))

Records after 2020: 0


,query,date,year,location,search_volume


## 4. Filter United States

In [7]:
us=df[df['location'].str.lower().isin(['united states','us','usa','united states of america'])]; print('US records:',len(us)); display(us[['query','date','location','search_volume']].head(20))

US records: 31804


,query,date,location,search_volume
0,shedeur sanders,NaT,United States,200.0
1,whoopi goldberg,NaT,United States,5000.0
2,mo williams,NaT,United States,5000.0
3,sloane stephens,NaT,United States,1000.0
4,canceled tv shows 2026,NaT,United States,5000.0
5,atlas - guadalajara,NaT,United States,50000.0
6,karoline leavitt,NaT,United States,2000.0
7,kali uchis,NaT,United States,2000.0
8,germany vs ghana,NaT,United States,2000.0
9,caleb flynn,NaT,United States,1000.0


## 5. Sort by Search Volume

In [8]:
sorted_df=df.sort_values('search_volume',ascending=False,na_position='last'); display(sorted_df[['query','date','year','location','search_volume']].head(20))

,query,date,year,location,search_volume
5442,chuck norris,NaT,NaN,United States,10000000.0
25024,savannah guthrie,NaT,NaN,United States,10000000.0
26437,catherine o'hara,NaT,NaN,United States,10000000.0
24439,2026 winter olympics,NaT,NaN,United States,10000000.0
22528,bad bunny halftime show,NaT,NaN,United States,10000000.0
19529,2026 winter olympics alpine skiing,NaT,NaN,United States,5000000.0
28538,rams vs seahawks,NaT,NaN,United States,5000000.0
22028,bad bunny,NaT,NaN,United States,5000000.0
23392,what time is the super bowl,NaT,NaN,United States,5000000.0
12797,2026 winter paralympics,NaT,NaN,United States,5000000.0


## 6. Mean, Median, Mode and Standard Deviation

In [9]:
v=df['search_volume'].dropna()
print('Mean =',v.mean())
print('Median =',v.median())
m=v.mode(); print('Mode =',m.iloc[0] if len(m) else 'No mode')
print('Standard Deviation =',v.std())

Mean = 27489.891208653
Median = 2000.0
Mode = 2000.0
Standard Deviation = 219937.68485987393


## 7. Descriptive Statistics

In [10]:
display(df['search_volume'].describe().to_frame().T)

,count,mean,std,min,25%,50%,75%,max
search_volume,31804.0,27489.891209,219937.68486,100.0,500.0,2000.0,10000.0,10000000.0


## 8. Group By Year – Average Search Volume

In [11]:
display(df.groupby('year')['search_volume'].mean().sort_index().to_frame('Average_Search_Volume'))

,Average_Search_Volume
year,


## 9. Group By Year – Number of Trends

In [12]:
display(df.groupby('year')['query'].count().sort_index().to_frame('Number_of_Trends'))

,Number_of_Trends
year,


## Final Summary

In [13]:
print('Unique queries:',df['query'].nunique()); print('Year range:',df['year'].min(),'to',df['year'].max()); print('Mean volume:',round(df['search_volume'].mean(),2)); display(sorted_df[['query','search_volume']].head(10))

Unique queries: 19709
Year range: nan to nan
Mean volume: 27489.89


,query,search_volume
5442,chuck norris,10000000.0
25024,savannah guthrie,10000000.0
26437,catherine o'hara,10000000.0
24439,2026 winter olympics,10000000.0
22528,bad bunny halftime show,10000000.0
19529,2026 winter olympics alpine skiing,5000000.0
28538,rams vs seahawks,5000000.0
22028,bad bunny,5000000.0
23392,what time is the super bowl,5000000.0
12797,2026 winter paralympics,5000000.0
